# RoadWatch – 02 Fine-tune (YOLOv8s, works on Colab and Kaggle)

Second experiment after the YOLOv8n baseline (`01_baseline.ipynb`, test mAP50 **0.342**).
Validation mAP50 was still rising at epoch 50, so this run uses a bigger model (**YOLOv8s**), more epochs
(**100**), a cosine learning-rate schedule and a few road-specific augmentations.

| ID | Class |
|---|---|
| 0 | D00 – Longitudinal crack |
| 1 | D10 – Transverse crack |
| 2 | D20 – Alligator crack |
| 3 | D40 – Pothole |

**Before you start**

- **Colab:** *Runtime → Change runtime type → T4 GPU*.
- **Kaggle:** *Settings → Accelerator → GPU T4 x2* (avoid P100 — recent PyTorch builds no longer support it),
  and *Settings → Internet → On* (needed for `pip` and `gdown`).

Then edit the **config cell** below (only that one) and run the cells from top to bottom.

## 1) Experiment config and environment

Change `RUN_NAME` for every new experiment — it is the output folder name, and the training cell refuses to
overwrite a run folder that already has a checkpoint.

In [1]:
# ===================== EXPERIMENT CONFIG – the only cell you should need to edit =====================
MODEL = 'yolov8s.pt'        # pretrained starting weights (n < s < m < l < x: bigger = slower but more accurate)
IMGSZ = 640                 # training / inference image size in pixels
EPOCHS = 100                # maximum number of epochs (early stopping may end training sooner)
BATCH = 16                  # images per batch (lower to 8 if you get "CUDA out of memory")
PATIENCE = 20               # stop early if val mAP has not improved for this many epochs
RUN_NAME = 'v8s_640_e100'   # output folder name – use a new name for every experiment
# =====================================================================================================

import os
import sys
from pathlib import Path

# Detect where the notebook is running.
if 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ:
    ENV = 'colab'
elif 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    ENV = 'kaggle'
else:
    ENV = 'local'

if ENV == 'colab':
    # Mount Drive so training runs survive a Colab disconnect.
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/RoadWatch/runs')
    WORK_DIR = Path('/content')            # dataset lives on the fast local disk, not on Drive
    OUT_DIR = Path('/content')             # final .pt and results zip are written here
elif ENV == 'kaggle':
    PROJECT = Path('/kaggle/working/runs') # /kaggle/working is kept as the notebook's output
    WORK_DIR = Path('/tmp/roadwatch')      # temporary disk – keeps the 774 MB dataset out of the output
    OUT_DIR = Path('/kaggle/working')
else:
    print('WARNING: neither Colab nor Kaggle detected – using local folders.')
    PROJECT = Path('runs').resolve()
    WORK_DIR = Path('work').resolve()
    OUT_DIR = WORK_DIR

for folder in (PROJECT, WORK_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Paths used by the rest of the notebook.
ZIP_PATH = WORK_DIR / 'RoadWatch_YOLO_Grouped.zip'
DATA_DIR = WORK_DIR / 'data'
DATA_YAML = WORK_DIR / 'data.yaml'
RUN_DIR = PROJECT / RUN_NAME               # exist_ok=True below keeps this name fixed (never RUN_NAME-2)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'
CLASS_NAMES = ['D00', 'D10', 'D20', 'D40']

print(f'Environment detected: {ENV.upper()}')
print(f'Runs folder (PROJECT): {PROJECT}')
print(f'This run:              {RUN_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment detected: COLAB
Runs folder (PROJECT): /content/drive/MyDrive/RoadWatch/runs
This run:              /content/drive/MyDrive/RoadWatch/runs/v8s_640_e100


## 2) Install packages and check the GPU

In [5]:
# Ultralytics provides YOLOv8; gdown downloads files from Google Drive links.
!pip install -q ultralytics gdown

import shutil
import subprocess
import torch

# Stop here with a clear message if no GPU is attached – training on a CPU would take days.
gpu_check = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if gpu_check is None or gpu_check.returncode != 0 or not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU found.\n'
        '  Colab:  Runtime -> Change runtime type -> T4 GPU\n'
        '  Kaggle: Settings -> Accelerator -> GPU T4 x2\n'
        'Then restart the session and run the notebook from the top.'
    )
print(gpu_check.stdout)

RuntimeError: No GPU found.
  Colab:  Runtime -> Change runtime type -> T4 GPU
  Kaggle: Settings -> Accelerator -> GPU T4 x2
Then restart the session and run the notebook from the top.

## 3) Download and unzip the dataset

Both steps are skipped if they were already done in this session.

> **If `gdown` fails** (quota exceeded / permission error):
> - **Colab:** open the shared file in Google Drive, choose **"Add shortcut to Drive"**, and run the fallback cell.
> - **Kaggle:** upload the zip as a Kaggle Dataset (*Add Input*). Kaggle unzips it automatically, so in the
>   fallback cell point `DATA_DIR` at the dataset's folder under `/kaggle/input/` and skip the unzip.

In [2]:
GDRIVE_FILE_ID = '1kIe0dMfRJWRbKxM2ZYB4_9GiU8pCYs7F'

# Download the grouped dataset zip from Google Drive (skip if it is already here).
if ZIP_PATH.exists():
    print(f'Zip already downloaded: {ZIP_PATH}')
else:
    subprocess.run(['gdown', GDRIVE_FILE_ID, '-O', str(ZIP_PATH)], check=True)

# Unzip into DATA_DIR (skip if already unzipped; delete DATA_DIR to force a fresh unzip).
if DATA_DIR.exists():
    print(f'Already unzipped: {DATA_DIR}')
else:
    print('Unzipping (takes a minute or two)...')
    shutil.unpack_archive(str(ZIP_PATH), str(DATA_DIR))
    print(f'Unzipped to {DATA_DIR}')

Zip already downloaded: /content/RoadWatch_YOLO_Grouped.zip
Already unzipped: /content/data


In [ ]:
# FALLBACK – only if gdown failed. Uncomment the lines for your platform.

# Colab: copy the zip from your Drive shortcut (adjust the path if it is in a sub-folder), then unzip.
# shutil.copy('/content/drive/MyDrive/RoadWatch_YOLO_Grouped.zip', ZIP_PATH)
# shutil.unpack_archive(str(ZIP_PATH), str(DATA_DIR))

# Kaggle: the uploaded dataset is already unzipped under /kaggle/input – just point DATA_DIR at it.
# DATA_DIR = Path('/kaggle/input/<your-dataset-name>')

## 4) Find the dataset root and check the splits

The zip may contain an extra top-level folder, so we search for the folder that contains `images/train`.
Expected image counts (from the dataset README): **train ≈ 5001, val ≈ 1429, test ≈ 714**.

In [3]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
EXPECTED = {'train': 5001, 'val': 1429, 'test': 714}

# The dataset root is the folder whose images/train sub-folder exists.
candidates = [p.parent.parent for p in DATA_DIR.rglob('images/train') if p.is_dir()]
if not candidates:
    raise FileNotFoundError(f'No images/train folder found under {DATA_DIR}. Did the unzip work?')
DATASET_ROOT = min(candidates, key=lambda p: len(p.parts))  # shallowest match
print('Dataset root:', DATASET_ROOT)

for split in ['train', 'val', 'test']:
    images = [f for f in (DATASET_ROOT / 'images' / split).iterdir() if f.suffix.lower() in IMAGE_EXTS]
    labels = list((DATASET_ROOT / 'labels' / split).glob('*.txt'))
    empty = sum(1 for f in labels if f.stat().st_size == 0)  # negative images have empty label files
    status = 'OK' if len(images) == EXPECTED[split] else f'WARNING: expected about {EXPECTED[split]}'
    print(f'{split:5s}: {len(images):5d} images | {len(labels):5d} labels ({empty} empty = negatives) | {status}')

# Sorted list of test images, used by the analysis and benchmark cells further down.
TEST_IMAGES = sorted(f for f in (DATASET_ROOT / 'images' / 'test').iterdir() if f.suffix.lower() in IMAGE_EXTS)

Dataset root: /content/data/RoadWatch_YOLO_Grouped
train:  5001 images |  5001 labels (2745 empty = negatives) | OK
val  :  1429 images |  1429 labels (784 empty = negatives) | OK
test :   714 images |   714 labels (392 empty = negatives) | OK


## 5) Write `data.yaml`

In [4]:
import yaml

data_config = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 4,
    'names': CLASS_NAMES,
}
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(DATA_YAML.read_text())

path: /content/data/RoadWatch_YOLO_Grouped
train: images/train
val: images/val
test: images/test
nc: 4
names:
- D00
- D10
- D20
- D40



## 6) Train

What changed compared with the baseline:

| Setting | Value | Why |
|---|---|---|
| `MODEL` | `yolov8s.pt` | ~3× more parameters than v8n; the baseline was still improving, so it has room to learn more. |
| `EPOCHS` / `PATIENCE` | 100 / 20 | More time to converge; early stopping still ends the run if val mAP plateaus. |
| `cos_lr=True` | cosine LR | The learning rate decays smoothly to its minimum instead of linearly, which usually helps the last epochs fine-tune. |
| `hsv_v=0.5` | brightness ±50 % | Road photos vary a lot in **lighting** (sun, shade, overcast, dusk). Randomly changing brightness teaches the model that a crack is still a crack in a shadow. (Default is 0.4.) |
| `fliplr=0.5` | horizontal flip, 50 % | A pothole on the left of the road looks like one on the right, so mirrored images are free extra data. (Same as the default; set explicitly so the experiment is documented.) Vertical flips are **not** used — the road is never upside down. |
| `degrees=5` | rotation ±5° | Simulates a slightly **tilted camera** or dashcam mount. Kept small because rotating an image makes the axis-aligned boxes a bit looser. (Default is 0.) |
| `close_mosaic=10` | mosaic off for the last 10 epochs | Mosaic (4 images stitched into one) helps early on, but the final epochs train on normal, full images like the ones the model will see in use. (Same as the default.) |

`exist_ok=True` writes into `RUN_DIR` exactly, never `RUN_NAME-2`. To avoid accidentally overwriting a finished
run, this cell does nothing if `RUN_DIR` already has a checkpoint — use the resume cell, or change `RUN_NAME`.

On a T4 expect very roughly 1.5–2 minutes per epoch for YOLOv8s at 640.

In [ ]:
from ultralytics import YOLO

if LAST_PT.exists():
    print(f'{RUN_DIR} already has a checkpoint ({LAST_PT.name}), so training was NOT started.')
    print('To continue that run, use the resume cell below. To start a new experiment, change RUN_NAME in the config cell.')
else:
    model = YOLO(MODEL)  # COCO-pretrained starting point
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        seed=42,                 # same seed as the baseline, for reproducibility
        project=str(PROJECT),
        name=RUN_NAME,
        exist_ok=True,           # always use RUN_DIR, never RUN_NAME-2
        cos_lr=True,             # cosine learning-rate schedule
        hsv_v=0.5,               # brightness augmentation (lighting)
        fliplr=0.5,              # horizontal flip probability
        degrees=5,               # random rotation of up to ±5 degrees
        close_mosaic=10,         # turn mosaic off for the last 10 epochs
    )
    print('Run saved to:', RUN_DIR)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data.yaml, degrees=5, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hs

## 7) Resume after a disconnect

Only needed if the session disconnected during training. It is safe to run at any time: it only resumes when the
run is really unfinished.

1. Reconnect with a GPU and **re-run the setup cells first: 1 (config), 2 (install/GPU), 3 (download/unzip),
   4 (dataset root) and 5 (data.yaml)**. The resumed run reads the dataset through `data.yaml`, so it must exist
   again at the same path. Do **not** re-run the training cell.
2. Run the cell below. Before calling `train()`, it checks:
   - **Is the run already finished?** A finished run's `last.pt` has `epoch = -1` and no optimizer state
     (also checked: `results.csv` has all epochs). Ultralytics cannot resume such a file — it would silently start
     a *new* training on its default `coco8` dataset — so the cell just prints *"Run already finished, nothing to
     resume"*.
   - **Does `data.yaml` exist, and is a GPU attached?** If not, it tells you what to re-run instead of training.
3. Otherwise it continues from `last.pt` with the original settings (epochs, augmentations, seed…), passing
   `data=DATA_YAML` (`/content/data.yaml` on Colab, `/tmp/roadwatch/data.yaml` on Kaggle) explicitly.

**Kaggle note:** `/kaggle/working` is wiped when an interactive session ends, so there is usually no `last.pt`
to resume from. For long runs on Kaggle use *Save Version → Save & Run All (Commit)*, which runs in the background
(up to 12 hours) and keeps `/kaggle/working` as the version's output.

In [ ]:
import csv
import torch
from ultralytics import YOLO


def training_status():
    # Return (finished, reason) for the run in RUN_DIR. Only reads files; never starts training.
    ckpt = torch.load(LAST_PT, map_location='cpu', weights_only=False)
    planned = (ckpt.get('train_args') or {}).get('epochs', EPOCHS)

    # 1) When training ends (all epochs done or early stopping), Ultralytics strips last.pt:
    #    'epoch' is set to -1 and the optimizer state is removed. Such a file cannot be resumed —
    #    Ultralytics would silently start a NEW training on its default dataset (coco8) instead.
    if ckpt.get('epoch', -1) == -1 or ckpt.get('optimizer') is None:
        return True, 'last.pt has no epoch/optimizer state – training ended (all epochs or early stopping)'

    # 2) Backup check: results.csv has one row per completed epoch.
    results_csv = RUN_DIR / 'results.csv'
    if results_csv.exists():
        with open(results_csv) as f:
            epochs_done = sum(1 for _ in csv.DictReader(f))
        if epochs_done >= planned:
            return True, f'results.csv already has {epochs_done} of {planned} epochs'

    return False, f'stopped after epoch {ckpt["epoch"] + 1} of {planned}'


if not LAST_PT.exists():
    print(f'No checkpoint at {LAST_PT} – nothing to resume.')
else:
    finished, reason = training_status()
    if finished:
        print(f'Run already finished, nothing to resume. ({reason})')
    elif not DATA_YAML.exists():
        print(f'{DATA_YAML} not found – re-run setup cells 3, 4 and 5 (download, dataset root, data.yaml) first.')
    elif not torch.cuda.is_available():
        print('No GPU in this session – switch the runtime to a T4 GPU, re-run the setup cells, then this cell.')
    else:
        print(f'Resuming {RUN_NAME}: {reason}.')
        print(f'Dataset: {DATA_YAML}')
        # data= is passed explicitly so the run never falls back to Ultralytics' default dataset.
        YOLO(str(LAST_PT)).train(resume=True, data=str(DATA_YAML))

## 8) Evaluate `best.pt` on the test split

The test split was not used for training or model selection, so these are the numbers to report.
Plots for the test split (PR curve, confusion matrix) are saved to `RUN_DIR/test/`, and the metrics to
`RUN_DIR/test_metrics.json` (same keys as the baseline JSON, plus a detailed `per_class` section).

In [8]:
import json
from ultralytics import YOLO

metrics = YOLO(str(BEST_PT)).val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUN_DIR),
    name='test',             # plots go to RUN_DIR/test/
    exist_ok=True,
)
box = metrics.box

print(f'mAP50     : {box.map50:.4f}')
print(f'mAP50-95  : {box.map:.4f}')
print(f'Precision : {box.mp:.4f}')
print(f'Recall    : {box.mr:.4f}')

# Per-class metrics. ap_class_index lists the classes found in the test split, in the same order as the
# per-class arrays (ap50, ap, p, r). A class with no test boxes would get None.
index_of = {int(class_id): i for i, class_id in enumerate(box.ap_class_index)}
per_class = {}
print(f"\n{'class':6s} {'AP50':>7s} {'AP50-95':>8s} {'P':>7s} {'R':>7s}")
for class_id, name in enumerate(CLASS_NAMES):
    i = index_of.get(class_id)
    if i is None:
        per_class[name] = {'AP50': None, 'AP50_95': None, 'precision': None, 'recall': None}
        print(f'{name:6s}     n/a (no boxes of this class in the test split)')
        continue
    per_class[name] = {
        'AP50': round(float(box.ap50[i]), 4),
        'AP50_95': round(float(box.ap[i]), 4),
        'precision': round(float(box.p[i]), 4),
        'recall': round(float(box.r[i]), 4),
    }
    m = per_class[name]
    print(f"{name:6s} {m['AP50']:7.4f} {m['AP50_95']:8.4f} {m['precision']:7.4f} {m['recall']:7.4f}")

# Same keys as the baseline's test_metrics.json, plus a 'per_class' section with all four numbers.
summary = {
    'weights': str(BEST_PT),
    'split': 'test',
    'mAP50': round(float(box.map50), 4),
    'mAP50_95': round(float(box.map), 4),
    'precision': round(float(box.mp), 4),
    'recall': round(float(box.mr), 4),
    'per_class_AP50': {name: m['AP50'] for name, m in per_class.items()},
    'per_class': per_class,
}
(RUN_DIR / 'test_metrics.json').write_text(json.dumps(summary, indent=2))
print('\nSaved', RUN_DIR / 'test_metrics.json')

Ultralytics 8.4.218 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1379.3±231.8 MB/s, size: 56.0 KB)
val: Scanning /content/data/RoadWatch_YOLO_Grouped/labels/test... 714 images, 392 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 714/714 2.1Kit/s 0.3s0.1ss
val: New cache created: /content/data/RoadWatch_YOLO_Grouped/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 9.4s/it 7:059.5ss
                   all        714        681      0.564      0.366      0.381      0.165
                   D00        101        144      0.464      0.292      0.303      0.125
                   D10          8          8       0.62      0.125      0.149      0.105
                   D20        190        211      0.678       0.64      0.664      0.279
                   D40        1

## 9) Compare all runs

Collects every `*/test_metrics.json` under `PROJECT` and adds the baseline numbers (copied from
`model_training/reports/baseline_v8n_test_metrics.json`, because the baseline run is not on Kaggle's disk).
If the baseline's own JSON is found under `PROJECT` (on Colab/Drive), it is not listed twice.

In [9]:
import json
import pandas as pd

# Baseline test results from 01_baseline (model_training/reports/baseline_v8n_test_metrics.json).
BASELINE = {
    'run': 'v8n_baseline (01_baseline)',
    'mAP50': 0.3415, 'mAP50_95': 0.1476, 'precision': 0.3706, 'recall': 0.387,
    'per_class_AP50': {'D00': 0.2811, 'D10': 0.0396, 'D20': 0.6546, 'D40': 0.3905},
}

runs = []
for metrics_file in sorted(PROJECT.glob('*/test_metrics.json')):
    result = json.loads(metrics_file.read_text())
    result['run'] = metrics_file.parent.name
    runs.append(result)

# Add the baseline unless the same numbers were already found on disk.
already_there = any(r['mAP50'] == BASELINE['mAP50'] and r['mAP50_95'] == BASELINE['mAP50_95'] for r in runs)
if not already_there:
    runs.append(BASELINE)

table = pd.DataFrame([
    {
        'run': r['run'],
        'mAP50': r['mAP50'],
        'mAP50-95': r['mAP50_95'],
        'P': r['precision'],
        'R': r['recall'],
        **{f'AP50 {name}': r.get('per_class_AP50', {}).get(name) for name in CLASS_NAMES},
    }
    for r in runs
]).sort_values('mAP50', ascending=False)

print(table.to_string(index=False, float_format='%.4f'))

           run  mAP50  mAP50-95      P      R  AP50 D00  AP50 D10  AP50 D20  AP50 D40
  v8s_640_e100 0.3807    0.1652 0.5641 0.3655    0.3025    0.1490    0.6636    0.4076
v8n_baseline-2 0.3415    0.1476 0.3706 0.3870    0.2811    0.0396    0.6546    0.3905


## 10) Missed-detection analysis

Runs `best.pt` on every test image and matches predictions to the ground-truth boxes. A ground-truth box counts as
**detected** if a prediction of the **same class** overlaps it with **IoU ≥ 0.5**; otherwise it is **missed**.
Each prediction can match at most one box (highest-confidence predictions are matched first).

Predictions below `CONF = 0.25` (Ultralytics' default) are ignored, so the miss counts depend on that threshold.

For each class, a grid of up to 6 test images with a missed box of that class is saved to
`RUN_DIR/failures/`: **green** = ground truth (thick green = missed), **red** = predictions with confidence.

In [ ]:
import gc
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

CONF = 0.25          # ignore predictions below this confidence
IOU_MATCH = 0.5      # minimum overlap for a prediction to count as finding a ground-truth box
N_EXAMPLES = 6       # example images per class in each grid
GC_EVERY = 100       # free memory and print progress every this many images
FAILURES_DIR = RUN_DIR / 'failures'
FAILURES_DIR.mkdir(parents=True, exist_ok=True)


def load_ground_truth(image_path, width, height):
    # Read the YOLO label file and convert normalised (xc, yc, w, h) to pixel (x1, y1, x2, y2).
    label_file = DATASET_ROOT / 'labels' / 'test' / f'{image_path.stem}.txt'
    boxes = []
    if label_file.exists():
        for line in label_file.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            xc, yc, w, h = (float(v) for v in parts[1:])
            boxes.append((class_id, (xc - w / 2) * width, (yc - h / 2) * height,
                          (xc + w / 2) * width, (yc + h / 2) * height))
    return boxes


def iou(a, b):
    # Intersection over union of two (x1, y1, x2, y2) boxes.
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


model = YOLO(str(BEST_PT))
total = {c: 0 for c in range(len(CLASS_NAMES))}    # ground-truth boxes per class
missed = {c: 0 for c in range(len(CLASS_NAMES))}   # missed boxes per class
records = []                                       # small records for images with at least one missed box

# Predict ONE image at a time and keep only small numpy copies of the boxes. The Results object (which holds
# the full image and tensors) is deleted straight away, so memory use stays flat over all test images.
for n, image_path in enumerate(TEST_IMAGES, start=1):
    results = model.predict(str(image_path), imgsz=IMGSZ, conf=CONF, verbose=False)
    result = results[0]
    height, width = result.orig_shape
    # .astype(..., copy=True) makes independent arrays, so nothing still points into the Results object.
    pred_cls = result.boxes.cls.cpu().numpy().astype(np.int32, copy=True)
    pred_xyxy = result.boxes.xyxy.cpu().numpy().astype(np.float32, copy=True)
    pred_conf = result.boxes.conf.cpu().numpy().astype(np.float32, copy=True)
    del results, result

    gt = load_ground_truth(image_path, width, height)

    # Greedy matching: highest-confidence predictions first, each to the best unmatched same-class box.
    matched = set()
    for k in np.argsort(-pred_conf):
        best_j, best_iou = None, IOU_MATCH
        for j, (gt_class, *gt_box) in enumerate(gt):
            if j in matched or gt_class != pred_cls[k]:
                continue
            overlap = iou(gt_box, pred_xyxy[k])
            if overlap >= best_iou:
                best_j, best_iou = j, overlap
        if best_j is not None:
            matched.add(best_j)

    missed_here = [j for j in range(len(gt)) if j not in matched]
    for gt_class, *_ in gt:
        total[gt_class] += 1
    for j in missed_here:
        missed[gt[j][0]] += 1
    if missed_here:
        # Only the file path and small arrays are kept – the image is re-read from disk when drawing.
        records.append({'path': image_path, 'gt': gt, 'missed': set(missed_here),
                        'pred_cls': pred_cls, 'pred_xyxy': pred_xyxy, 'pred_conf': pred_conf})

    if n % GC_EVERY == 0 or n == len(TEST_IMAGES):
        gc.collect()
        print(f'  processed {n}/{len(TEST_IMAGES)} images ({len(records)} with missed boxes so far)')

# Summary table.
print(f"\n{'class':6s} {'GT boxes':>9s} {'detected':>9s} {'missed':>7s} {'miss rate':>10s}")
for c, name in enumerate(CLASS_NAMES):
    rate = missed[c] / total[c] if total[c] else 0.0
    print(f'{name:6s} {total[c]:9d} {total[c] - missed[c]:9d} {missed[c]:7d} {rate:10.1%}')


def draw(record):
    # Re-read the image from disk and draw ground truth in green (thick if missed) and predictions in red.
    image = cv2.cvtColor(cv2.imread(str(record['path'])), cv2.COLOR_BGR2RGB)
    for j, (gt_class, x1, y1, x2, y2) in enumerate(record['gt']):
        thickness = 5 if j in record['missed'] else 2
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), thickness)
        cv2.putText(image, f'GT {CLASS_NAMES[gt_class]}', (int(x1), max(15, int(y1) - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    for pred_class, (x1, y1, x2, y2), conf in zip(record['pred_cls'], record['pred_xyxy'], record['pred_conf']):
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (220, 0, 0), 2)
        cv2.putText(image, f'{CLASS_NAMES[pred_class]} {conf:.2f}', (int(x1), min(image.shape[0] - 5, int(y2) + 18)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (220, 0, 0), 2)
    return image


# One grid per class: up to N_EXAMPLES test images that contain a missed box of that class.
for c, name in enumerate(CLASS_NAMES):
    examples = [r for r in records if any(r['gt'][j][0] == c for j in r['missed'])][:N_EXAMPLES]
    if not examples:
        print(f'{name}: no missed boxes – no grid saved.')
        continue
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax in axes.flat:
        ax.axis('off')
    for ax, record in zip(axes.flat, examples):
        ax.imshow(draw(record))
        ax.set_title(record['path'].name, fontsize=9)
    fig.suptitle(f'{RUN_NAME} – missed {name} (green = ground truth, thick = missed; red = prediction)', fontsize=14)
    fig.tight_layout()
    out_file = FAILURES_DIR / f'missed_{name}.png'
    fig.savefig(out_file, dpi=100)
    plt.show()
    plt.close(fig)   # release the figure and its images before drawing the next grid
    gc.collect()
    print('Saved', out_file)

## 11) CPU benchmark (PyTorch and ONNX)

Times end-to-end inference (pre-processing + model + post-processing) on the **CPU** for 50 test images, one at a
time, after 5 warm-up images. Then exports `best.pt` to ONNX and times that the same way.
Results go to `RUN_DIR/cpu_benchmark.json`.

Note: Colab and Kaggle give you only a few slow virtual CPUs, so these numbers are mainly useful for
**comparing runs with each other**, not as the speed on the final deployment hardware.

In [5]:
import json
import os
import statistics
import time
import torch
from ultralytics import YOLO

N_BENCH = 50    # images to time
WARMUP = 5      # untimed warm-up images (the first runs are always slower)
bench_images = [str(p) for p in TEST_IMAGES[:N_BENCH]]


def time_on_cpu(weights_path):
    # Return per-image times in milliseconds for the given weights file on the CPU.
    model = YOLO(str(weights_path), task='detect')
    for image in bench_images[:WARMUP]:
        model.predict(image, imgsz=IMGSZ, device='cpu', verbose=False)
    times = []
    for image in bench_images:
        start = time.perf_counter()
        model.predict(image, imgsz=IMGSZ, device='cpu', verbose=False)
        times.append((time.perf_counter() - start) * 1000)
    return times


def summarise(label, weights_path, times):
    size_mb = Path(weights_path).stat().st_size / (1024 * 1024)
    result = {
        'file': str(weights_path),
        'size_mb': round(size_mb, 2),
        'ms_per_image_mean': round(statistics.mean(times), 1),
        'ms_per_image_median': round(statistics.median(times), 1),
    }
    print(f"{label:8s}: {result['ms_per_image_mean']:7.1f} ms/image (median {result['ms_per_image_median']:.1f}) | {size_mb:.2f} MB")
    return result


# Which CPU are we on? (Recorded so numbers from different machines are not mixed up.)
cpu_name = 'unknown'
if Path('/proc/cpuinfo').exists():
    for line in Path('/proc/cpuinfo').read_text().splitlines():
        if line.startswith('model name'):
            cpu_name = line.split(':', 1)[1].strip()
            break
print(f'CPU: {cpu_name} | {os.cpu_count()} logical cores | PyTorch threads: {torch.get_num_threads()}\n')

pytorch_result = summarise('PyTorch', BEST_PT, time_on_cpu(BEST_PT))

# Export to ONNX (written next to best.pt as best.onnx; may install the onnx packages the first time).
onnx_path = Path(YOLO(str(BEST_PT)).export(format='onnx', imgsz=IMGSZ))
onnx_result = summarise('ONNX', onnx_path, time_on_cpu(onnx_path))

benchmark = {
    'run': RUN_NAME,
    'environment': ENV,
    'cpu': cpu_name,
    'cpu_logical_cores': os.cpu_count(),
    'torch_threads': torch.get_num_threads(),
    'imgsz': IMGSZ,
    'images_timed': len(bench_images),
    'warmup_images': WARMUP,
    'pytorch': pytorch_result,
    'onnx': onnx_result,
}
(RUN_DIR / 'cpu_benchmark.json').write_text(json.dumps(benchmark, indent=2))
print('\nSaved', RUN_DIR / 'cpu_benchmark.json')

CPU: Intel(R) Xeon(R) CPU @ 2.20GHz | 2 logical cores | PyTorch threads: 1

PyTorch :   552.4 ms/image (median 493.4) | 21.48 MB
Ultralytics 8.4.218 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/RoadWatch/runs/v8s_640_e100/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (21.5 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.18 environment at: /usr
Resolved 12 packages in 336ms
Prepared 4 packages in 621ms
Installed 4 packages in 39ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.32.0
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 1.4s
WARNI

## 12) Export the model and a results bundle

- Copies `best.pt` to `roadwatch_v1_<RUN_NAME>.pt`.
- Zips the results, each renamed with a `<RUN_NAME>_` prefix: `test_metrics.json`, `results.png` (training
  curves), the **test-split** `confusion_matrix_normalized.png` and PR curve, the failure grids and `cpu_benchmark.json`.
- **Colab:** downloads both files to your computer. **Kaggle:** prints their paths under `/kaggle/working`
  (download them from the notebook's *Output* panel).

Copy the unzipped files into `model_training/reports/` in the repo. The `.pt` file is git-ignored — keep it on Drive.

In [6]:
import shutil
import zipfile

# 1) Copy the best weights under a release-style name.
final_pt = OUT_DIR / f'roadwatch_v1_{RUN_NAME}.pt'
shutil.copy2(BEST_PT, final_pt)

# 2) Collect the result files. The test-split plots are in RUN_DIR/test; fall back to the training (val) plots.
test_plots = RUN_DIR / 'test'
confusion = test_plots / 'confusion_matrix_normalized.png'
if not confusion.exists():
    confusion = RUN_DIR / 'confusion_matrix_normalized.png'
# The PR-curve file is called PR_curve.png or BoxPR_curve.png depending on the Ultralytics version.
pr_curves = sorted(test_plots.glob('*PR_curve.png')) or sorted(RUN_DIR.glob('*PR_curve.png'))

files_to_zip = {
    'test_metrics.json': RUN_DIR / 'test_metrics.json',
    'results.png': RUN_DIR / 'results.png',
    'confusion_matrix_normalized.png': confusion,
    'PR_curve.png': pr_curves[0] if pr_curves else RUN_DIR / 'PR_curve.png',
    'cpu_benchmark.json': RUN_DIR / 'cpu_benchmark.json',
}
for grid in sorted((RUN_DIR / 'failures').glob('*.png')):
    files_to_zip[f'failures_{grid.name}'] = grid

# 3) Write the zip, prefixing every file with RUN_NAME. Missing files are reported, not fatal.
zip_path = OUT_DIR / f'{RUN_NAME}_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name, source in files_to_zip.items():
        if source.exists():
            zf.write(source, arcname=f'{RUN_NAME}_{name}')
            print(f'  added   {RUN_NAME}_{name}')
        else:
            print(f'  MISSING {source} (did you run the cell that creates it?)')

print(f'\nModel: {final_pt} ({final_pt.stat().st_size / 1024 / 1024:.1f} MB)')
print(f'Zip:   {zip_path}')

# 4) Download (Colab) or show where the files are (Kaggle).
if ENV == 'colab':
    from google.colab import files
    files.download(str(zip_path))
    files.download(str(final_pt))
else:
    print('\nDownload these from the Output panel (Kaggle) or copy them from the paths above.')

  added   v8s_640_e100_test_metrics.json
  added   v8s_640_e100_results.png
  added   v8s_640_e100_confusion_matrix_normalized.png
  added   v8s_640_e100_PR_curve.png
  added   v8s_640_e100_cpu_benchmark.json

Model: /content/roadwatch_v1_v8s_640_e100.pt (21.5 MB)
Zip:   /content/v8s_640_e100_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
!cp /content/v8s_640_e100_results.zip /content/roadwatch_v1_v8s_640_e100.pt /content/drive/MyDrive/RoadWatch/